# Let's work on this task: 

Correlate vaccination coverage with a non-vaccine-preventable health outcome.

A strong association with an unrelated outcome may indicate confounding, spurious correlation, or problems in the analysis pipeline.

### Let's use
- app/data/tycho_measles_control.csv, Tycho Measles
- raw/Vaccination_Coverage_and_Exemptions_among_Kindergartners_20260827.csv

In [1]:
import pandas as pd

cases_df = pd.read_csv('../app/data/tycho_cases.csv')
cases_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases
0,1995,AK,NaN,13.0,1.0
1,1995,AL,NaN,4.0,38.0
2,1995,AR,2.0,10.0,41.0
3,1995,AZ,10.0,2.0,151.0
4,1995,CA,108.0,206.0,463.0


In [2]:
coverage_df = pd.read_csv('../app/data/nis_vacc_coverage.csv')
coverage_df.head()

,state,year,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated
0,AK,1995,89.86,194.0,177.0,89.86,194.0,177.0,77.58,194.0,159.0
1,AK,1996,84.55,260.0,225.0,84.55,260.0,225.0,78.25,260.0,210.0
2,AK,1997,87.41,291.0,257.0,87.41,291.0,257.0,80.98,291.0,242.0
3,AK,1998,87.06,34.0,30.0,87.06,34.0,30.0,82.01,34.0,28.0
4,AK,1999,90.67,349.0,321.0,90.67,349.0,321.0,83.54,349.0,296.0


In [3]:
panel_df = pd.merge(cases_df, coverage_df, on=['year', 'state'])
panel_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated
0,1995,AK,NaN,13.0,1.0,89.86,194.0,177.0,89.86,194.0,177.0,77.58,194.0,159.0
1,1995,AL,NaN,4.0,38.0,88.77,419.0,386.0,88.77,419.0,386.0,78.93,419.0,357.0
2,1995,AR,2.0,10.0,41.0,90.43,237.0,218.0,90.43,237.0,218.0,77.24,237.0,190.0
3,1995,AZ,10.0,2.0,151.0,82.29,372.0,319.0,82.29,372.0,319.0,74.52,372.0,296.0
4,1995,CA,108.0,206.0,463.0,90.32,681.0,627.0,90.32,681.0,627.0,76.76,681.0,542.0


# Here are examples by ChatGPT for non-vaccine related events
| Placebo outcome                    | Event/death count | State | Year | Good for state-year analysis? |
| ---------------------------------- | :---------------: | :---: | :--: | ----------------------------- |
| **Motor vehicle traffic deaths**   |         ✅         |   ✅   |   ✅  | ⭐ **Excellent**               |
| **Unintentional injury deaths**    |         ✅         |   ✅   |   ✅  | ⭐ **Excellent**               |
| **Drowning deaths**                |         ✅         |   ✅   |   ✅  | ✅ Good                        |
| **Accidental firearm deaths**      |         ✅         |   ✅   |   ✅  | ⚠️ Many small counts          |
| **Kidney stone deaths**            |         ✅         |   ✅   |   ✅  | ⚠️ Likely sparse              |
| **Appendicitis deaths**            |         ✅         |   ✅   |   ✅  | ⚠️ Likely sparse              |
| **Falls / accidental fall deaths** |         ✅         |   ✅   |   ✅  | ✅ Good                        |
| **Cancer deaths**                  |         ✅         |   ✅   |   ✅  | ⭐ **Excellent**               |


## Let's do cancer deaths
- 1995 to 1998: https://wonder.cdc.gov/cmf-icd9.html?utm_source=chatgpt.com
- 1999 to 2017: https://wonder.cdc.gov/cancer-v2022.html

In [4]:
import pandas as pd
import us

cancer_1995_1998_df = pd.read_csv("../raw/cdc/Compressed Mortality, 1979-1998.csv")
cancer_1995_1998_df = cancer_1995_1998_df[['Year', "State", "Deaths"]]
cancer_1995_1998_df.dropna(inplace=True)


cancer_1995_1998_df["State"] = cancer_1995_1998_df["State"].apply(
    lambda x: us.states.lookup(x).abbr if us.states.lookup(x) else None
)

cancer_1995_1998_df['Year'] = cancer_1995_1998_df['Year'].astype(int)
cancer_1995_1998_df.rename(columns={'Year': 'year', 'State': 'state', 'Deaths': 'cancer_death_count'}, inplace=True)
cancer_1995_1998_df.head()

,year,state,cancer_death_count
0,1995,AL,9414.0
1,1995,AK,574.0
2,1995,AZ,8020.0
3,1995,AR,6079.0
4,1995,CA,51423.0


In [5]:
cancer_1999_2017_df = pd.read_csv("../raw/cdc/United States and Puerto Rico Cancer Statistics, 1999-2022 Incidence.csv")
cancer_1999_2017_df = cancer_1999_2017_df[['States', 'Year', 'Count']]
cancer_1999_2017_df.dropna(inplace=True)


cancer_1999_2017_df["States"] = cancer_1999_2017_df["States"].apply(
    lambda x: us.states.lookup(x).abbr if us.states.lookup(x) else None
)

cancer_1999_2017_df['Year'] = cancer_1999_2017_df['Year'].astype(int)
cancer_1999_2017_df.rename(columns={'Year': 'year', 'States': 'state', 'Count': 'cancer_death_count'}, inplace=True)
cancer_1999_2017_df.head()

,state,year,cancer_death_count
0,AL,1999,19894.0
1,AL,2000,20462.0
2,AL,2001,21738.0
3,AL,2002,22134.0
4,AL,2003,21477.0


In [6]:
cancer_1995_1998_df['year'].min(), cancer_1995_1998_df['year'].max()

(np.int64(1995), np.int64(1998))

In [7]:
cancer_1999_2017_df['year'].min(), cancer_1999_2017_df['year'].max()

(np.int64(1999), np.int64(2017))

In [8]:
concat_cancer_deaths = pd.concat([cancer_1995_1998_df, cancer_1999_2017_df], ignore_index=True)

In [9]:
concat_cancer_deaths.head()

,year,state,cancer_death_count
0,1995,AL,9414.0
1,1995,AK,574.0
2,1995,AZ,8020.0
3,1995,AR,6079.0
4,1995,CA,51423.0


In [10]:
concat_cancer_deaths['year'].min(), concat_cancer_deaths['year'].max()

(np.int64(1995), np.int64(2017))

In [11]:
len(concat_cancer_deaths)

1167

In [12]:
concat_cancer_deaths.to_csv('../app/data/cancer_deaths.csv', index=False)

# Ok lets graph cancer deaths with vacc coverage

In [13]:
vacc_coverage_df = pd.read_csv('../app/data/nis_vacc_coverage.csv')
vacc_coverage_df.head()

,state,year,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated
0,AK,1995,89.86,194.0,177.0,89.86,194.0,177.0,77.58,194.0,159.0
1,AK,1996,84.55,260.0,225.0,84.55,260.0,225.0,78.25,260.0,210.0
2,AK,1997,87.41,291.0,257.0,87.41,291.0,257.0,80.98,291.0,242.0
3,AK,1998,87.06,34.0,30.0,87.06,34.0,30.0,82.01,34.0,28.0
4,AK,1999,90.67,349.0,321.0,90.67,349.0,321.0,83.54,349.0,296.0


In [14]:
cancer_deaths_df = pd.read_csv('../app/data/cancer_deaths.csv')
cancer_deaths_df.head()

,year,state,cancer_death_count
0,1995,AL,9414.0
1,1995,AK,574.0
2,1995,AZ,8020.0
3,1995,AR,6079.0
4,1995,CA,51423.0


In [15]:
panel_df = pd.merge(cancer_deaths_df, vacc_coverage_df, on=['year', 'state'])
panel_df.head()

,year,state,cancer_death_count,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated
0,1995,AL,9414.0,88.77,419.0,386.0,88.77,419.0,386.0,78.93,419.0,357.0
1,1995,AK,574.0,89.86,194.0,177.0,89.86,194.0,177.0,77.58,194.0,159.0
2,1995,AZ,8020.0,82.29,372.0,319.0,82.29,372.0,319.0,74.52,372.0,296.0
3,1995,AR,6079.0,90.43,237.0,218.0,90.43,237.0,218.0,77.24,237.0,190.0
4,1995,CA,51423.0,90.32,681.0,627.0,90.32,681.0,627.0,76.76,681.0,542.0


In [19]:
panel_df[panel_df['state'] == 'WA']

,year,state,cancer_death_count,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated
46,1995,WA,9938.0,89.73,451.0,412.0,89.73,451.0,412.0,81.49,451.0,382.0
96,1996,WA,10063.0,91.71,553.0,517.0,91.71,553.0,517.0,81.09,553.0,460.0
146,1997,WA,10035.0,91.21,619.0,569.0,91.14,619.0,568.0,83.62,619.0,528.0
196,1998,WA,10277.0,94.24,62.0,59.0,94.24,62.0,59.0,88.51,62.0,56.0
1068,1999,WA,29110.0,89.47,616.0,557.0,89.28,616.0,556.0,80.94,616.0,511.0
1069,2000,WA,29083.0,90.23,570.0,509.0,90.23,570.0,509.0,82.60,570.0,471.0
1070,2001,WA,30399.0,89.55,592.0,539.0,89.35,592.0,538.0,79.94,592.0,484.0
1071,2002,WA,30725.0,89.84,548.0,499.0,89.59,548.0,497.0,77.39,548.0,429.0
1072,2003,WA,30929.0,93.54,580.0,549.0,93.47,580.0,548.0,83.69,580.0,503.0
1073,2004,WA,31884.0,92.30,548.0,511.0,92.30,548.0,511.0,84.98,548.0,480.0


In [24]:
import plotly.graph_objects as go


def graph_cancer_vaccination_coverage(df, state):

    # Filter to selected state
    state_df = (
        df[df["state"] == state]
        .sort_values("year")
    )

    fig = go.Figure()

    # Cancer deaths
    fig.add_trace(
        go.Bar(
            x=state_df["year"],
            y=state_df["cancer_death_count"],
            name="Cancer deaths",
            opacity=0.65,
            hovertemplate=(
                "Year: %{x}<br>"
                "Cancer deaths: %{y:,.0f}"
                "<extra></extra>"
            )
        )
    )

    # Measles coverage
    fig.add_trace(
        go.Scatter(
            x=state_df["year"],
            y=state_df["measles_coverage_pct"],
            name="Measles coverage",
            mode="lines+markers",
            yaxis="y2",
            hovertemplate=(
                "Year: %{x}<br>"
                "Measles coverage: %{y:.2f}%"
                "<extra></extra>"
            )
        )
    )

    # Mumps coverage
    fig.add_trace(
        go.Scatter(
            x=state_df["year"],
            y=state_df["mumps_coverage_pct"],
            name="Mumps coverage",
            mode="lines+markers",
            yaxis="y2",
            hovertemplate=(
                "Year: %{x}<br>"
                "Mumps coverage: %{y:.2f}%"
                "<extra></extra>"
            )
        )
    )

    # Pertussis coverage
    fig.add_trace(
        go.Scatter(
            x=state_df["year"],
            y=state_df["pertussis_coverage_pct"],
            name="Pertussis coverage",
            mode="lines+markers",
            yaxis="y2",
            hovertemplate=(
                "Year: %{x}<br>"
                "Pertussis coverage: %{y:.2f}%"
                "<extra></extra>"
            )
        )
    )

    fig.update_layout(
        title=f"Annual Vaccination Coverage and Cancer Deaths In {state}",

        xaxis=dict(
            title="Year",
            dtick=1
        ),

        yaxis=dict(
            title="Cancer Death Count"
        ),

        yaxis2=dict(
            title="Vaccination Coverage (%)",
            overlaying="y",
            side="right",
            range=[0, 100]
        ),

        template="plotly_white",
        height=600,
        hovermode="x unified",
        legend_title="Measure",
        bargap=0.20
    )

    fig.show()

In [30]:
graph_cancer_vaccination_coverage(panel_df, 'CA')